In [ ]:
%run "./00_CC_Mapping_Setup.ipynb" 

In [ ]:
%sql
/* ===================================================================================
   METRIC: TP04 - TP Sole Source
   
   BUSINESS LOGIC: 
   - Denominator: N/A (Numerical Count).
   - Numerator: No of the unit's active third party vendors who are single source 
     AND located in high risk jurisdictions.
   - Rule 1: Use Single Source file to bridge Engagement ID to ThirdPartyName.
   - Rule 2: Col D, M, N: if any have high risk country, counted as 1.
   - Rule 3: Col K & L format can be comma-separated. Expand rows and remove duplicates.
   
   ARCHITECTURE SUMMARY:
   1. MASTER AU ANCHOR: Output strictly controlled by the filtered Master AU list.
   2. SINGLE SOURCE BRIDGE: Filters published_contracts for 'Single Source'.
   3. FLATTEN LOBs: Uses LATERAL VIEW explode(split(...)) to expand comma-separated 
      lob_using and owning_lob strings into individual evaluation rows.
   4. SAFE MAPPING: Joins expanded LOBs using standard LIKE syntax.
   5. DEDUPLICATION: Uses COUNT(DISTINCT EngagementNumber) grouped by AU_ID.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Mapped_AUs AS (
    SELECT DISTINCT 
        TRIM(CAST(Assessable_Unit_ID AS STRING)) AS AU_ID,
        TRIM(TPRM_Units) AS TPRM_Units
    FROM hive_metastore.ra_adido_2025.third_party_unit_mapping
    WHERE Assessable_Unit_ID IS NOT NULL
      AND NULLIF(TRIM(TPRM_Units), '') IS NOT NULL
),

High_Risk_Countries AS (
    SELECT UPPER(TRIM(Jurisdiction)) AS CountryName
    FROM hive_metastore.ra_adido_2025.td_country_risk_rating_abac 
    WHERE TRIM(FinalABACRating) = 'High'
),

Single_Source_Engagements AS (
    -- Bridge the true Engagement ID from the Single Source file
    SELECT DISTINCT TRIM(ThirdPartyName) AS True_Engagement_ID
    FROM hive_metastore.ra_adido_2025.published_contrcats_19_dec_2025
    WHERE UPPER(TRIM(EngagementNumber)) = 'SINGLE SOURCE'
      AND ThirdPartyName IS NOT NULL
),

Third_Party_Risk_Data AS (
    SELECT 
        p.EngagementNumber,
        p.owning_lob,
        p.lob_using,
        p.location_of_tp,
        p.country_prod_serv_originates,
        p.Td_Countries_Impacted,
        CASE WHEN COALESCE(TRIM(p.location_of_tp), '') = ''
              AND COALESCE(TRIM(p.country_prod_serv_originates), '') = ''
              AND COALESCE(TRIM(p.Td_Countries_Impacted), '') = '' THEN 1
             ELSE 0 END AS Is_Missing_Jurisdiction
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3 p
    INNER JOIN Single_Source_Engagements s
        ON TRIM(p.EngagementNumber) = s.True_Engagement_ID
),

High_Risk_Engagements AS (
    -- Strictly filter for High Risk countries or Missing Jurisdictions
    SELECT DISTINCT 
        tp.EngagementNumber,
        tp.owning_lob,
        tp.lob_using
    FROM Third_Party_Risk_Data tp
    LEFT JOIN High_Risk_Countries h1 ON UPPER(TRIM(tp.location_of_tp)) = h1.CountryName
    LEFT JOIN High_Risk_Countries h2 ON UPPER(TRIM(tp.country_prod_serv_originates)) = h2.CountryName
    LEFT JOIN High_Risk_Countries h3 ON UPPER(TRIM(tp.Td_Countries_Impacted)) = h3.CountryName
    WHERE h1.CountryName IS NOT NULL
       OR h2.CountryName IS NOT NULL
       OR h3.CountryName IS NOT NULL
       OR tp.Is_Missing_Jurisdiction = 1
),

Flattened_LOBs AS (
    -- FLATTEN RULE: Expand comma-separated values into individual rows
    SELECT 
        EngagementNumber, 
        TRIM(exploded_lob) AS Expanded_LOB
    FROM High_Risk_Engagements
    LATERAL VIEW explode(split(owning_lob, ',')) AS exploded_lob
    WHERE exploded_lob IS NOT NULL AND TRIM(exploded_lob) != ''
    
    UNION
    
    SELECT 
        EngagementNumber, 
        TRIM(exploded_lob) AS Expanded_LOB
    FROM High_Risk_Engagements
    LATERAL VIEW explode(split(lob_using, ',')) AS exploded_lob
    WHERE exploded_lob IS NOT NULL AND TRIM(exploded_lob) != ''
),

Engagements_By_AU AS (
    -- Map expanded rows to AU using SAFE LIKE logic
    SELECT 
        m.AU_ID,
        -- DEDUPLICATION RULE: Count distinct engagements, not flattened rows
        COUNT(DISTINCT f.EngagementNumber) AS High_Risk_Count
    FROM Flattened_LOBs f
    INNER JOIN Mapped_AUs m
        ON UPPER(f.Expanded_LOB) LIKE '%' || UPPER(m.TPRM_Units) || '%'
    GROUP BY m.AU_ID
)

SELECT 
    a.BusinessID,                          
    a.AU_Name,                             
    a.Business_Segment,
    'TP04' AS QuestionID,               
    COALESCE(CAST(e.High_Risk_Count AS STRING), '0') AS Response,
    a.In_ABAC_AU_List
    
FROM Master_AUs a
LEFT JOIN Engagements_By_AU e 
    ON a.BusinessID = e.AU_ID
ORDER BY a.BusinessID;

In [ ]:
%sql
WITH High_Risk_Countries AS (
    SELECT UPPER(TRIM(Jurisdiction)) AS CountryName
    FROM hive_metastore.ra_adido_2025.td_country_risk_rating_abac 
    WHERE TRIM(FinalABACRating) = 'High'
),

Single_Source_Engagements AS (
    SELECT DISTINCT TRIM(ThirdPartyName) AS True_Engagement_ID
    FROM hive_metastore.ra_adido_2025.published_contrcats_19_dec_2025
    WHERE UPPER(TRIM(EngagementNumber)) = 'SINGLE SOURCE'
      AND ThirdPartyName IS NOT NULL
),

Mapped_AUs AS (
    SELECT DISTINCT 
        TRIM(CAST(Assessable_Unit_ID AS STRING)) AS AU_ID,
        TRIM(TPRM_Units) AS TPRM_Units
    FROM hive_metastore.ra_adido_2025.third_party_unit_mapping
    WHERE Assessable_Unit_ID IS NOT NULL
      AND NULLIF(TRIM(TPRM_Units), '') IS NOT NULL
),

Filtered_Engagements AS (
    SELECT 
        tp.EngagementNumber,
        tp.owning_lob,
        tp.lob_using,
        tp.location_of_tp,
        tp.country_prod_serv_originates,
        tp.Td_Countries_Impacted
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3 tp
    INNER JOIN Single_Source_Engagements s
        ON TRIM(tp.EngagementNumber) = s.True_Engagement_ID
    LEFT JOIN High_Risk_Countries h1 ON UPPER(TRIM(tp.location_of_tp)) = h1.CountryName
    LEFT JOIN High_Risk_Countries h2 ON UPPER(TRIM(tp.country_prod_serv_originates)) = h2.CountryName
    LEFT JOIN High_Risk_Countries h3 ON UPPER(TRIM(tp.Td_Countries_Impacted)) = h3.CountryName
    WHERE h1.CountryName IS NOT NULL
       OR h2.CountryName IS NOT NULL
       OR h3.CountryName IS NOT NULL
       OR (COALESCE(TRIM(tp.location_of_tp), '') = ''
           AND COALESCE(TRIM(tp.country_prod_serv_originates), '') = ''
           AND COALESCE(TRIM(tp.Td_Countries_Impacted), '') = '')
),

Flattened_LOBs AS (
    SELECT 
        EngagementNumber, 
        owning_lob AS Original_String,
        TRIM(exploded_lob) AS Expanded_LOB,
        'owning_lob' AS Source_Column
    FROM Filtered_Engagements
    LATERAL VIEW explode(split(owning_lob, ',')) AS exploded_lob
    WHERE exploded_lob IS NOT NULL AND TRIM(exploded_lob) != ''
    
    UNION ALL
    
    SELECT 
        EngagementNumber, 
        lob_using AS Original_String,
        TRIM(exploded_lob) AS Expanded_LOB,
        'lob_using' AS Source_Column
    FROM Filtered_Engagements
    LATERAL VIEW explode(split(lob_using, ',')) AS exploded_lob
    WHERE exploded_lob IS NOT NULL AND TRIM(exploded_lob) != ''
)

SELECT DISTINCT
    COALESCE(map.AU_ID, 'UNMAPPED_ENGAGEMENT') AS BusinessID,
    f.EngagementNumber,
    f.Expanded_LOB,
    f.Source_Column,
    map.TPRM_Units AS Matched_Mapping_String,
    f.Original_String AS Full_Comma_Separated_List

FROM Flattened_LOBs f
LEFT JOIN Mapped_AUs map
    ON UPPER(f.Expanded_LOB) LIKE '%' || UPPER(map.TPRM_Units) || '%'
ORDER BY BusinessID, f.EngagementNumber;